In [7]:
import pandas as pd
import sys
from pathlib import Path

# Add the project root directory (amazon) to Python path
root_dir = Path.cwd().parents[2]  # Moves up 3 levels from notebooks/khalid
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
pd.set_option("future.infer_string", False)
from amazon.code.business_entity_resolution.src.dataio import load, load_gt, load_pairs
from amazon.code.business_entity_resolution.src.norm_missingness import add_flags, check_flags, summary
from amazon.code.business_entity_resolution.src.norm_text import add_clean, check_clean, clean
from amazon.code.business_entity_resolution.src.norm_script import add_latin
from amazon.code.business_entity_resolution.src.norm_keys import build_locality_vocab, add_keys, check_keys, profile
CACHE = Path("../../cache")

In [2]:
s1, s2, s3 = load("train")      # pandas, cached after first call
gt = load_gt("train")   # one row per S1 entity, singletons kept

In [ ]:
n1, n2, n3 = len(s1), len(s2), len(s3)
s1, s2, s3 = add_flags(s1), add_flags(s2), add_flags(s3)

for df, lbl, n in [(s1,"S1",n1), (s2,"S2",n2), (s3,"S3",n3)]:
    check_flags(df, lbl, n_rows=n)
pd.concat([summary(s1,"S1"), summary(s2,"S2"), summary(s3,"S3")], axis=1)

[missingness/S1] ok (25 checks)
[missingness/S2] ok (25 checks)
[missingness/S3] ok (25 checks)


,S1,S2,S3
rows,2206821.0,5034616.00,5285603.00
blank_address,0.0,168967.00,175916.00
blank_address_pct,0.0,3.36,3.33
addr_degenerate,0.0,0.00,0.00
no_name,0.0,6.00,18.00
name_short,0.0,699.00,9257.00
unmatchable,0.0,0.00,0.00


In [13]:
s1 = add_clean(s1); check_clean(s1, "S1")
s2 = add_clean(s2); check_clean(s2, "S2")
s3 = add_clean(s3); check_clean(s3, "S3")
#s2[["business_name","name_clean","business_address","addr_clean"]].sample(15)

s1.to_parquet(CACHE / "train_source1_norm.parquet")
s2.to_parquet(CACHE / "train_source2_norm.parquet")
s3.to_parquet(CACHE / "train_source3_norm.parquet")

[text/S1] ok (10 checks)
[text/S2] ok (11 checks)
[text/S3] ok (11 checks)


In [2]:
st1, st2, st3 = load("test")

In [3]:
st3 = add_flags(st3)

In [17]:
n1, n2, n3 = len(st1), len(st2), len(st3)
st1, st2, st3 = add_flags(st1), add_flags(st2), add_flags(st3)

for df, lbl, n in [(st1,"S1",n1), (st2,"S2",n2), (st3,"S3",n3)]:
    check_flags(df, lbl, n_rows=n)

pd.concat([summary(st1,"S1"), summary(st2,"S2"), summary(st3,"S3")], axis=1)

[missingness/S1] ok (25 checks)
[missingness/S2] ok (25 checks)
[missingness/S3] ok (25 checks)


,S1,S2,S3
rows,1732544.0,4887273.00,5082316.00
blank_address,0.0,129408.00,136098.00
blank_address_pct,0.0,2.65,2.68
addr_degenerate,0.0,0.00,0.00
no_name,0.0,49.00,61.00
name_short,0.0,10117.00,19850.00
unmatchable,0.0,0.00,0.00


In [ ]:
st1 = add_clean(st1); check_clean(st1, "St1")
st1.to_parquet(CACHE / "test_source1_norm.parquet")

st2 = add_clean(st2); check_clean(st2, "St2")
st2.to_parquet(CACHE / "test_source2_norm.parquet")

[text/St1] ok (10 checks)
[text/St2] ok (11 checks)


AssertionError: [text/St3] 1 issue(s):
  - name: cleaning did not empty a populated field -- got 1, want 0

In [20]:
had = st3["business_name"].str.strip() != ""
st3.loc[had & (st3["name_clean"] == ""), ["entity_id", "business_name"]]

,entity_id,business_name
4048578,S3-648724017,Null


In [4]:
st3 = add_clean(st3); check_clean(st3, "St3")
st3.to_parquet(CACHE / "test_source3_norm.parquet")

[text/St3] ok (11 checks)


In [14]:
for n, df in [("S2", s2), ("S3", s3)]:
    m = df["business_name"].str.contains("[\u0980-\u09FF]", regex=True, na=False)
    print(n, "bengali:", round(m.mean() * 100, 3), "%")

S2 bengali: 0.61 %
S3 bengali: 0.343 %


In [2]:
# loading cached checkpoint training data
s1 = pd.read_parquet(CACHE / "train_source1_norm.parquet")
s2 = pd.read_parquet(CACHE / "train_source2_norm.parquet")
s3 = pd.read_parquet(CACHE / "train_source3_norm.parquet")
# loading cached checkpoint test data
st1 = pd.read_parquet(CACHE / "test_source1_norm.parquet")
st2 = pd.read_parquet(CACHE / "test_source2_norm.parquet")
st3 = pd.read_parquet(CACHE / "test_source3_norm.parquet")

In [8]:
# loading cached checkpoint test data
st1 = pd.read_parquet(CACHE / "test_source1_norm.parquet")
st2 = pd.read_parquet(CACHE / "test_source2_norm.parquet")
st3 = pd.read_parquet(CACHE / "test_source3_norm.parquet")

In [7]:
import unicodedata
from collections import Counter

sample = s2.loc[s2["business_name"].str.contains("[^\x00-\x7F]", regex=True, na=False),
                "business_name"].sample(20000, random_state=0)
blocks = Counter(unicodedata.name(c, "?").split()[0]
                 for s in sample for c in s if ord(c) > 127)
print(blocks.most_common(15))

[('DEVANAGARI', 182360), ('TELUGU', 30427), ('KANNADA', 28987), ('TAMIL', 27253), ('BENGALI', 22372), ('GUJARATI', 20502), ('MALAYALAM', 14457), ('LATIN', 7962), ('ORIYA', 5305), ('GURMUKHI', 4000), ('ZERO', 780)]


In [8]:
sample = st2.loc[st2["business_name"].str.contains("[^\x00-\x7F]", regex=True, na=False),
                "business_name"].sample(20000, random_state=0)
blocks = Counter(unicodedata.name(c, "?").split()[0]
                 for s in sample for c in s if ord(c) > 127)
print(blocks.most_common(15))

[('DEVANAGARI', 174333), ('TELUGU', 29358), ('KANNADA', 28708), ('TAMIL', 27565), ('BENGALI', 21458), ('GUJARATI', 20535), ('MALAYALAM', 14992), ('LATIN', 9160), ('ORIYA', 5737), ('GURMUKHI', 3798), ('ZERO', 787)]


In [9]:
sample = st1.loc[st1["business_name"].str.contains("[^\x00-\x7F]", regex=True, na=False),
                "business_name"].sample(20000, random_state=0)
blocks = Counter(unicodedata.name(c, "?").split()[0]
                 for s in sample for c in s if ord(c) > 127)
print(blocks.most_common(15))

[('LATIN', 23110), ('?', 1)]


In [10]:
sample = st3.loc[st3["business_name"].str.contains("[^\x00-\x7F]", regex=True, na=False),
                "business_name"].sample(20000, random_state=0)
blocks = Counter(unicodedata.name(c, "?").split()[0]
                 for s in sample for c in s if ord(c) > 127)
print(blocks.most_common(15))

[('DEVANAGARI', 121849), ('KANNADA', 21372), ('TELUGU', 21061), ('TAMIL', 18417), ('BENGALI', 15240), ('GUJARATI', 14144), ('LATIN', 12408), ('MALAYALAM', 11443), ('ORIYA', 3925), ('GURMUKHI', 3094), ('ZERO', 624)]


In [11]:
st1[st1["business_name"].str.contains("[^\x00-\x7F]", regex=True, na=False)].head()

,entity_id,business_name,business_address,country,has_address,addr_degenerate,addr_usable,has_name,name_short,name_clean,addr_clean
36,S1-385801270,Maison de Santé Generation,"30 Rue Louis Thénard, Saint-Nazaire, Pays de l...",France,True,False,True,True,False,maison de sante generation,30 rue louis thenard saint nazaire pays de la ...
70,S1-295346696,École primaire Sainte Pierre,"22 RUE Descartes, Hauts-de-France, Calais",France,True,False,True,True,False,ecole primaire sainte pierre,22 rue descartes hauts de france calais
191,S1-527611593,Établissements Demployeurs SARL,"234 Rue de la République, Dunkerque, Hauts-de-...",France,True,False,True,True,False,etablissements demployeurs sarl,234 rue de la republique dunkerque hauts de fr...
225,S1-693956169,École primaire de la Sainte,"841 Avenue de Dunkerque, Lille, Hauts-de-France",France,True,False,True,True,False,ecole primaire de la sainte,841 avenue de dunkerque lille hauts de france
309,S1-871112769,École primaire du Hj,"36 Rue des Trois Rois, Nantes, Pays de la Loire",France,True,False,True,True,False,ecole primaire du hj,36 rue des trois rois nantes pays de la loire


In [3]:
indic = s2["name_clean"].str.contains("[\u0900-\u0D7F]", regex=True, na=False)

dev2 = pd.concat([
    s2.sample(50_000, random_state=0),
    s2[indic].sample(10_000, random_state=0),
]).drop_duplicates(subset="entity_id")

In [5]:
from amazon.code.business_entity_resolution.src.norm_script import add_latin, check_latin, script_profile

s1 = add_latin(s1); check_latin(s1, "S1")
s1.to_parquet(CACHE / "train_source1_norm.parquet")

s2 = add_latin(s2); check_latin(s2, "S2")
s1.to_parquet(CACHE / "train_source2_norm.parquet")

s3 = add_latin(s3); check_latin(s3, "S3")
s3.to_parquet(CACHE / "train_source3_norm.parquet")

[script/S1] ok (8 checks)
[script/S2] ok (8 checks)
[script/S3] ok (8 checks)


In [13]:

#st1 = add_latin(st1); check_latin(st1, "St1")
st1.to_parquet(CACHE / "test_source1_norm.parquet")

#st2 = add_latin(st2); check_latin(st2, "St2")
st2.to_parquet(CACHE / "test_source2_norm.parquet")

#st3 = add_latin(st3); check_latin(st3, "St3")
st3.to_parquet(CACHE / "test_source3_norm.parquet")

In [11]:
script_profile(s2)   # expect ~9% non-latin, matching EDA §1.6


name_script
latin         0.9058
devanagari    0.0535
telugu        0.0078
kannada       0.0074
tamil         0.0067
gujarati      0.0061
bengali       0.0061
malayalam     0.0037
oriya         0.0015
gurmukhi      0.0013
Name: proportion, dtype: float64

In [12]:
script_profile(st2)

name_script
latin         0.8882
devanagari    0.0632
telugu        0.0092
kannada       0.0090
tamil         0.0080
bengali       0.0073
gujarati      0.0072
malayalam     0.0045
oriya         0.0018
gurmukhi      0.0016
Name: proportion, dtype: float64

In [5]:
from rapidfuzz import fuzz
from amazon.code.business_entity_resolution.src.norm_script import _to_latin

from anyascii import anyascii
pairs = load_pairs("train")
# S2 records that are Indic and have a known S1 match
cand = s2.loc[indic, ["entity_id", "name_clean"]].merge(
    pairs, left_on="entity_id", right_on="mid")
cand = cand.merge(s1[["entity_id", "name_clean"]],
                  left_on="source1_entity_id", right_on="entity_id",
                  suffixes=("_s2", "_s1")).sample(2000, random_state=0)

for fn, lbl in [(_to_latin, "IAST"), (anyascii, "anyascii")]:
    out = clean(cand["name_clean_s2"].map(fn))
    print(lbl, round(sum(fuzz.token_set_ratio(a, b)
                         for a, b in zip(out, cand["name_clean_s1"])) / len(cand), 1))

IAST 68.0
anyascii 73.3


In [7]:
import re
REPAIRS = [(r"ph", "f"), (r"m(?=[kgcjtdnpbm])", "n")]

def repaired(text):
    s = anyascii(text)
    for pat, rep in REPAIRS:
        s = re.sub(pat, rep, s)
    return re.sub(r"(\w{3,}?)a\b", r"\1", s)   # trailing schwa

out = clean(cand["name_clean_s2"].map(repaired))
print("anyascii+repairs", round(sum(fuzz.token_set_ratio(a, b)
      for a, b in zip(out, cand["name_clean_s1"])) / len(cand), 1))

variants = {
    "raw":    lambda t: anyascii(t),
    "ph->f":  lambda t: re.sub(r"ph", "f", anyascii(t)),
    "schwa":  lambda t: re.sub(r"(\w{3,}?)a\b", r"\1", anyascii(t)),
    "all":    repaired,
}
for lbl, fn in variants.items():
    out = clean(cand["name_clean_s2"].map(fn))
    print(lbl, round(sum(fuzz.token_set_ratio(a, b)
          for a, b in zip(out, cand["name_clean_s1"])) / len(cand), 1))

anyascii+repairs 75.2
raw 73.3
ph->f 74.5
schwa 73.1
all 75.2


In [14]:
allnames = pd.concat([d["name_latin"] for d in (s1, s2, s3, st1, st2, st3)])
toks = allnames.str.split()
print(toks.str[-1].value_counts().head(40).to_dict())

{'limited': 4225834, 'llc': 2114138, 'ltd': 1780895, 'inc': 1457651, 'com': 724362, 'center': 601060, 'private': 517837, 'corp': 482618, 'co': 458422, 'services': 404981, 'partners': 387809, 'llp': 377019, 'group': 370611, 'sarl': 325033, 'holdings': 261871, 'sas': 238352, 'corporation': 210915, 'pc': 209417, 'service': 184459, 'lp': 156311, 'pvt': 155295, 'li': 135021, 'company': 115716, 'associates': 115328, 'pllc': 101853, 'care': 96148, 'elelpi': 95652, 'eurl': 94775, 'limitet': 94539, 'clinic': 88396, 'sa': 77548, 'enterprises': 70284, 'sasu': 69180, 'incorporated': 66929, 'sci': 59810, 'trust': 56114, 'and': 56103, 'ventures': 55657, 'industries': 54631, 'limirrd': 53534}


In [15]:
allnames[allnames.str.endswith(" com")].sample(20).tolist()

['wisdominstitutetechnology com',
 'rrv project pvt ltd www rrvproje com',
 'tirupatimediatech com',
 'privateshivamlogistics com',
 'ponceterra com',
 'sfsena com',
 'clinicwilmington com',
 'dreher secure www drehersec com',
 'smt privategeneticsone com',
 'nationalchoicetwin com',
 'intelligenttrust com',
 'lyricwells com',
 'stierwaltsclearlending com',
 'houstonpatriotplum com',
 'sandeselectrical com',
 'williamswebsterhagerstown com',
 'nineagarwal com',
 'shri techventures com',
 'newcastleaurora com',
 'amicale du com']

In [16]:
t = toks.str[-1].value_counts()
print(t[t.index.str.startswith("lim")].head(20).to_dict())
print(t[t.index.str.startswith("priv")].head(10).to_dict())

{'limited': 4225834, 'limitet': 94539, 'limirrd': 53534, 'limted': 53, 'lima': 50, 'lim': 44, 'lime': 30, 'limtied': 19, 'limied': 14, 'limon': 14, 'limerick': 9, 'limousine': 9, 'limoux': 9, 'limes': 8, 'limits': 8, 'limitless': 7, 'limousin': 7, 'limlted': 6, 'limoon': 5, 'limaque': 5}
{'private': 517837, 'privee': 471, 'prive': 100, 'privat': 15, 'privte': 10, 'privated': 8, 'privette': 8, 'privilege': 7, 'prives': 7, 'privite': 6}


#### Re iterate and clean

In [5]:
import polars as pl
for split in ("train", "test"):
    for n, df in enumerate(load(split), start=1):
        df = add_latin(add_clean(add_flags(df)))
        pl.from_pandas(df).write_parquet(CACHE / f"{split}_source{n}_norm.parquet")
        print(split, n, "done", len(df))

train 1 done 2206821
train 2 done 5034616
train 3 done 5285603
test 1 done 1732544
test 2 done 4887273
test 3 done 5082316


In [12]:
# loading cached checkpoint training data
s1 = pd.read_parquet(CACHE / "train_source1_norm.parquet")
s2 = pd.read_parquet(CACHE / "train_source2_norm.parquet")
s3 = pd.read_parquet(CACHE / "train_source3_norm.parquet")
# loading cached checkpoint test data
st1 = pd.read_parquet(CACHE / "test_source1_norm.parquet")
st2 = pd.read_parquet(CACHE / "test_source2_norm.parquet")
st3 = pd.read_parquet(CACHE / "test_source3_norm.parquet")

In [ ]:
vocab = build_locality_vocab([s1, s2, s3, st1, st2, st3])
vocab = {w for w in vocab if len(w) >= 4}
print(len(vocab), sorted(vocab)[:40])

2516 ['aarey', 'abdul', 'aberdeen', 'above', 'abul', 'academy', 'acharya', 'acres', 'acton', 'adams', 'adarsh', 'addison', 'aditya', 'adyar', 'agarwal', 'aggarwal', 'agra', 'agrahara', 'ahmadabad', 'ahmed', 'ahmedabad', 'ahmednagar', 'airoli', 'airport', 'airy', 'ajay', 'ajit', 'ajmer', 'akola', 'akron', 'alabama', 'alappuzha', 'alaska', 'albans', 'albany', 'albemarle', 'albert', 'albuquerque', 'alexander', 'alexandria']


In [13]:
indic = s2["name_script"] != "latin"

dev2 = pd.concat([
    s2.sample(50_000, random_state=0),
    s2[indic].sample(10_000, random_state=0),
]).drop_duplicates(subset="entity_id")

dev2 = add_keys(dev2, vocab)
check_keys(dev2, "S2-dev")
dev2[["name_latin", "name_core", "suffix_set", "initialism",
      "addr_numbers", "addr_tokens", "addr_locality"]].sample(10)

[keys/S2-dev] ok (19 checks)


,name_latin,name_core,suffix_set,initialism,addr_numbers,addr_tokens,addr_locality
114763,shri dec0 engineering pvt ltd,shri dec0 engineering,ltd pvt,sde,58,alwar katori tibara tijara wala,alwar wala
2444120,delhi ltd center,delhi center,ltd,dc,302,delhi east kailash new,delhi east kailash
4768927,fiva best biotechnology inc,fiva best biotechnology,inc,fbb,948,athletic morrisville,
1016649,gujrat prodyusr praivet limited,gujrat prodyusr praivet,limited,gpp,1 101,agwanpur faridabad gali haryana kalan kheri na...,faridabad gali haryana kalan kheri nagar
4512972,tirupti aintrpraij ij praivet lintid,tirupti aintrpraij ij praivet lintid,,taipl,107 1st,complex ganj kismat ludhiana miller,complex ganj ludhiana miller
479620,apex wellness,apex wellness,,aw,8300,bainbridge loop olympia,loop olympia
114200,siix chicago group,siix chicago group,,scg,501,glen jefferson rose,glen jefferson rose
2135708,balaji investments limited private,balaji investments,limited private,bi,5,apt banglow dombivli gokul jyot karve maharash...,dombivli gokul karve maharashtra west
4024817,cascade beacno atlanticus care,cascade beacno atlanticus care,,cbac,1208,place raleigh weldon,place raleigh
2994254,cascade services,cascade services,,cs,,jones opelika robert trent trl,jones robert


In [14]:
frames = {"train_source1": s1, "train_source2": s2, "train_source3": s3,
          "test_source1": st1, "test_source2": st2, "test_source3": st3}

for name, df in frames.items():
    out = add_keys(df, vocab)
    check_keys(out, name)
    pl.from_pandas(out).write_parquet(CACHE / f"{name}_keys.parquet")
    frames[name] = out
    print(name, "done")

[keys/train_source1] ok (19 checks)
train_source1 done
[keys/train_source2] ok (19 checks)
train_source2 done
[keys/train_source3] ok (19 checks)
train_source3 done
[keys/test_source1] ok (19 checks)
test_source1 done
[keys/test_source2] ok (19 checks)
test_source2 done
[keys/test_source3] ok (19 checks)
test_source3 done


In [15]:
import json
pd.concat([profile(df, n) for n, df in frames.items()], axis=1)
json.dump(sorted(vocab), open(CACHE / "locality_vocab.json", "w"))

### Testing pipeline

In [16]:
d = pd.read_parquet(CACHE / "test_source3_keys.parquet")
print(d.shape, list(d.columns))

(5082316, 20) ['entity_id', 'business_name', 'business_address', 'country', 'has_address', 'addr_degenerate', 'addr_usable', 'has_name', 'name_short', 'name_clean', 'addr_clean', 'name_script', 'name_latin', 'name_core', 'name_tokens', 'suffix_set', 'initialism', 'addr_numbers', 'addr_tokens', 'addr_locality']


In [17]:
from pathlib import Path
import amazon.code.business_entity_resolution.src.dataio as dataio
import amazon.code.business_entity_resolution.src.normalize_pipeline as npl

tmp = Path("cache_test"); tmp.mkdir(exist_ok=True)
dataio.CACHE = tmp
npl.CACHE = tmp

npl.run(splits=("train",), refresh=True)

phase 1 -- flags, clean, transliterate
[missingness/train_source1] ok (25 checks)
[text/train_source1] ok (10 checks)
[script/train_source1] ok (8 checks)
  train_source1: 2,206,821 rows
    scripts {'latin': 1.0}
[missingness/train_source2] ok (25 checks)


KeyboardInterrupt: 